In [13]:
from __future__ import annotations

from dataclasses import dataclass, field
from pathlib import Path
from typing import Dict, Optional, Sequence, Tuple, Union

import glob
import numpy as np
import xarray as xr
import xcdat as xc
import pandas as pd

from ExperimentUtil import (
    run_dict,
    open_var_from_files,
    get_comp_dir,
    find_nc_files,
    collect_variable_list,
    flatten_variable_dict,
    open_compvar,
    set_monthly_time_to_midmonth_cftime
)

In [16]:
@dataclass(frozen=True)
class MjoConfig:
    time_name: str = "time"
    lat_name: str = "lat"
    lon_name: str = "lon"
    lat_band: Tuple[float, float] = (-10.0, 10.0)
    lon_sector: Tuple[float, float] = (60.0, 150.0)
    bandpass_days: Tuple[float, float] = (20.0, 100.0)
    lags: np.ndarray = field(default_factory=lambda: np.arange(-20, 21))


# -----------------------------
# Utilities
# -----------------------------
def with_time_index(
    da: xr.DataArray,
    *,
    time_name: str = "time",
    time_str_name: str = "time_str",
) -> xr.DataArray:
    """
    Replace time coord with integer day index (0..N-1) to avoid calendar conflicts.
    Store original time strings in a uniquely-named coord (time_str_name) to avoid merge conflicts.
    Safe to call multiple times on a DataArray.
    """
    # Drop groupby helper coords + any existing time_str coord to avoid merge conflicts
    drop_names = [n for n in ["dayofyear", time_str_name] if n in da.coords]
    if drop_names:
        da = da.drop_vars(drop_names, errors="ignore")

    if time_name not in da.coords:
        raise ValueError(f"with_time_index: '{time_name}' not found in DataArray coords: {list(da.coords)}")

    n = da.sizes[time_name]
    time_str = xr.DataArray(
        [str(t) for t in da[time_name].values],
        dims=(time_name,),
        name=time_str_name,
    )

    out = da.assign_coords({time_name: np.arange(n)})
    out = out.assign_coords({time_str_name: time_str})
    return out

def should_flip_to_match_ref(mjo, model_field, ref_field, anomaly="daily_climatology") -> bool:
    """
    Decide sign flip using correlation of lon-mean standardized hov anomalies.
    Works across xarray versions that do NOT support skipna= in xr.corr().
    Assumes model_field/ref_field are already time-sliced upstream.
    """
    hov_m = mjo.hovmoller_from_da(
        model_field, lon_sector=mjo.cfg.lon_sector, anomaly=anomaly, standardize=True
    )
    hov_r = mjo.hovmoller_from_da(
        ref_field,   lon_sector=mjo.cfg.lon_sector, anomaly=anomaly, standardize=True
    )

    a = hov_m.mean(mjo.lon_name, skipna=True) if mjo.lon_name in hov_m.dims else hov_m
    b = hov_r.mean(mjo.lon_name, skipna=True) if mjo.lon_name in hov_r.dims else hov_r

    # Align + mask invalids (skipna substitute)
    a0, b0 = xr.align(a, b, join="inner")
    valid = np.isfinite(a0) & np.isfinite(b0)
    c = xr.corr(a0.where(valid), b0.where(valid), dim=mjo.cfg.time_name)

    return float(c) < 0.0

# -----------------------------
# Preprocessor
# -----------------------------
class Preprocessor:
    def __init__(self, lat_band: Tuple[float, float] = (-10.0, 10.0)):
        self.lat_band = lat_band

    def normalize_lon_360(self, da: xr.DataArray, lon_name: str = "lon") -> xr.DataArray:
        """Ensure longitude is in [0, 360) and sorted. Safe to call multiple times."""
        if lon_name not in da.coords:
            return da

        lon = da[lon_name]
        try:
            lon_min = float(lon.min().compute())
        except Exception:
            lon_min = float(lon.min())

        if lon_min < 0.0:
            da = da.assign_coords({lon_name: (lon % 360)})

        return da.sortby(lon_name)

    def subset_latlon(
        self,
        da: xr.DataArray,
        *,
        lat_name: str = "lat",
        lon_name: str = "lon",
        lon_sector: Optional[Tuple[float, float]] = None,
    ) -> xr.DataArray:
        # lon normalization first
        da = self.normalize_lon_360(da, lon_name=lon_name)

        # ensure latitude slicing works even if lat descending
        if lat_name in da.coords:
            da = da.sortby(lat_name)

        lat0, lat1 = self.lat_band
        da2 = da.sel({lat_name: slice(lat0, lat1)})

        if da2.sizes.get(lat_name, 0) == 0:
            lat0_in = float(da[lat_name].min())
            lat1_in = float(da[lat_name].max())
            raise ValueError(
                f"Empty lat selection: {lat_name}=slice({lat0},{lat1}). "
                f"Input lat range [{lat0_in:.1f},{lat1_in:.1f}]."
            )

        if lon_sector is not None:
            lo0, lo1 = lon_sector
            da2 = da2.sel({lon_name: slice(lo0, lo1)})

            if da2.sizes.get(lon_name, 0) == 0:
                lon0_in = float(da[lon_name].min())
                lon1_in = float(da[lon_name].max())
                raise ValueError(
                    f"Empty lon selection: {lon_name}=slice({lo0},{lo1}). "
                    f"Input lon range [{lon0_in:.1f},{lon1_in:.1f}] after lon normalization."
                )

        return da2

    def lat_mean(self, da: xr.DataArray, lat_name: str = "lat") -> xr.DataArray:
        weights = np.cos(np.deg2rad(da[lat_name]))
        weights = weights / weights.mean()
        return da.weighted(weights).mean(lat_name)

    def daily_climatology_anomaly(self, da: xr.DataArray, *, time_name: str = "time") -> xr.DataArray:
        clim = da.groupby(f"{time_name}.dayofyear").mean(time_name)
        return da.groupby(f"{time_name}.dayofyear") - clim

    def standardize(self, da: xr.DataArray, *, dim: str = "time") -> xr.DataArray:
        mu = da.mean(dim, skipna=True)
        sd = da.std(dim, skipna=True)
        z = (da - mu) / sd
        z = z.where(np.isfinite(z), 0.0)
        z = z.where(sd > 0, 0.0)
        return z

    def bandpass_fft(
        self,
        da: xr.DataArray,
        *,
        low_days: float,
        high_days: float,
        dim: str = "time",
    ) -> xr.DataArray:
        """
        Simple FFT bandpass along time.
        Assumes daily data and evenly spaced time.
        NOTE: This forces compute (uses numpy FFT).
        """
        x = da.transpose(dim, ...)
        n = x.sizes[dim]

        freqs = np.fft.rfftfreq(n, d=1.0)  # cycles/day
        low_f = 1.0 / high_days
        high_f = 1.0 / low_days

        X = np.fft.rfft(x.values, axis=0)
        mask = (freqs >= low_f) & (freqs <= high_f)

        if X.ndim == 1:
            Xf = X * mask
        else:
            Xf = X * mask[(slice(None),) + (None,) * (X.ndim - 1)]

        xf = np.fft.irfft(Xf, n=n, axis=0)
        out = xr.DataArray(xf, dims=x.dims, coords=x.coords, name=getattr(da, "name", None))
        return out.transpose(*da.dims)


# -----------------------------
# Metrics engine
# -----------------------------
class MJOMetrics:
    def __init__(
        self,
        cfg: Optional[MjoConfig] = None,
        *,
        lat_band: Tuple[float, float] = (-10.0, 10.0),
        lat_name: str = "lat",
        lon_name: str = "lon",
    ):
        self.cfg = cfg or MjoConfig(lat_name=lat_name, lon_name=lon_name, lat_band=lat_band)
        self.lat_name = self.cfg.lat_name
        self.lon_name = self.cfg.lon_name
        self.time_name = self.cfg.time_name
        self.pp = Preprocessor(lat_band=self.cfg.lat_band)

    # -------- coordinate normalization --------
    def _normalize_latlon_names(self, da: xr.DataArray) -> xr.DataArray:
        """
        Make sure spatial dims match cfg.lat_name/cfg.lon_name.
        Handles ERA5 (latitude/longitude) and drops singleton vertical dims.
        """
        rename_map = {}

        if self.lat_name not in da.dims:
            for cand in ["latitude", "nav_lat", "y", "LAT", "Latitude"]:
                if cand in da.dims:
                    rename_map[cand] = self.lat_name
                    break

        if self.lon_name not in da.dims:
            for cand in ["longitude", "nav_lon", "x", "LON", "Longitude"]:
                if cand in da.dims:
                    rename_map[cand] = self.lon_name
                    break

        if rename_map:
            da = da.rename(rename_map)

        for lev in ["level", "lev", "plev"]:
            if lev in da.dims and da.sizes.get(lev, 0) == 1:
                da = da.isel({lev: 0}, drop=True)

        return da

    # -------- time slicing --------
    def _safe_time_slice(self, da: xr.DataArray, start: str, end: str) -> xr.DataArray:
        tname = self.time_name
        if tname not in da.coords:
            return da

        try:
            da = da.sortby(tname)
        except Exception:
            pass

        t0 = da[tname].values[0]
        y0, m0, d0 = map(int, start.split("-"))
        y1, m1, d1 = map(int, end.split("-"))

        if isinstance(t0, (np.datetime64,)):
            return da.sel({tname: slice(np.datetime64(start), np.datetime64(end))})

        dt_cls = type(t0)
        try:
            s = dt_cls(y0, m0, d0)
            e = dt_cls(y1, m1, d1)
            return da.sel({tname: slice(s, e)})
        except Exception:
            return da.sel({tname: slice(start, end)})

    # -------- lag correlation --------
    def _lagcorr_1d(self, a: xr.DataArray, b: xr.DataArray, lags: np.ndarray) -> xr.DataArray:
        a0, b0 = xr.align(a, b, join="inner")
        valid = np.isfinite(a0) & np.isfinite(b0)
        a0 = a0.where(valid)
        b0 = b0.where(valid)
    
        vals = []
        for lag in lags:
            lag_i = int(lag)
            if lag_i > 0:
                aa = a0.isel({self.time_name: slice(None, -lag_i)})
                bb = b0.isel({self.time_name: slice(lag_i, None)})
            elif lag_i < 0:
                aa = a0.isel({self.time_name: slice(-lag_i, None)})
                bb = b0.isel({self.time_name: slice(None, lag_i)})
            else:
                aa, bb = a0, b0
    
            # NO skipna kwarg (compat across xarray versions)
            vals.append(xr.corr(aa, bb, dim=self.time_name))
    
        return xr.DataArray(vals, dims=("lag",), coords={"lag": lags}, name="lagcorr")

    # -------- shared preprocessing --------
    def prep_ts_only(
        self,
        da: xr.DataArray,
        *,
        lon_sector: Optional[Tuple[float, float]] = None,
        bandpass_days: Optional[Tuple[float, float]] = None,
        standardize: bool = True,
        time_range: Optional[Tuple[str, str]] = None,
    ) -> xr.DataArray:
        if lon_sector is None:
            lon_sector = self.cfg.lon_sector
        if bandpass_days is None:
            bandpass_days = self.cfg.bandpass_days

        low, high = bandpass_days

        if time_range is not None:
            da = self._safe_time_slice(da, time_range[0], time_range[1])

        da = self._normalize_latlon_names(da)

        da = self.pp.subset_latlon(da, lat_name=self.lat_name, lon_name=self.lon_name, lon_sector=lon_sector)
        da = self.pp.lat_mean(da, lat_name=self.lat_name)
        ts = da.mean(self.lon_name, skipna=True)  # (time,)

        ts = self.pp.daily_climatology_anomaly(ts, time_name=self.time_name)
        ts = self.pp.bandpass_fft(ts, low_days=low, high_days=high, dim=self.time_name)

        if standardize:
            ts = self.pp.standardize(ts, dim=self.time_name)

        return ts

    # alias kept for backwards compatibility
    def prep_series(self, da: xr.DataArray, **kwargs) -> xr.DataArray:
        return self.prep_ts_only(da, **kwargs)

    # -------- hovmoller --------
    def hovmoller_from_da(
        self,
        da: xr.DataArray,
        *,
        start: Optional[str] = None,
        end: Optional[str] = None,
        lon_sector: Optional[Tuple[float, float]] = None,
        anomaly: str = "daily_climatology",
        standardize: bool = True,
    ) -> xr.DataArray:
        if start is not None or end is not None:
            if start is None or end is None:
                raise ValueError("Provide both start and end if slicing in hovmoller_from_da.")
            da = self._safe_time_slice(da, start, end)

        da = self._normalize_latlon_names(da)

        da = self.pp.subset_latlon(
            da, lat_name=self.lat_name, lon_name=self.lon_name, lon_sector=lon_sector
        )
        da = self.pp.lat_mean(da, lat_name=self.lat_name)  # (time, lon)

        if anomaly == "daily_climatology":
            da = self.pp.daily_climatology_anomaly(da, time_name=self.time_name)
        elif anomaly == "none":
            pass
        else:
            raise ValueError(f"Unknown anomaly method: {anomaly}")

        if standardize:
            da = self.pp.standardize(da, dim=self.time_name)

        return da

    # -------- coupling --------
    def coupling_from_da(
        self,
        conv_da: xr.DataArray,
        wind_da: xr.DataArray,
        *,
        lon_sector: Optional[Tuple[float, float]] = None,
        bandpass_days: Optional[Tuple[float, float]] = None,
        lags: Optional[np.ndarray] = None,
        standardize: bool = True,
        time_range: Optional[Tuple[str, str]] = None,
        sign_align_ref_conv: Optional[xr.DataArray] = None,
    ) -> xr.DataArray:
        if lon_sector is None:
            lon_sector = self.cfg.lon_sector
        if bandpass_days is None:
            bandpass_days = self.cfg.bandpass_days
        if lags is None:
            lags = self.cfg.lags

        conv_ts = self.prep_ts_only(
            conv_da,
            lon_sector=lon_sector,
            bandpass_days=bandpass_days,
            standardize=standardize,
            time_range=time_range,
        )
        wind_ts = self.prep_ts_only(
            wind_da,
            lon_sector=lon_sector,
            bandpass_days=bandpass_days,
            standardize=standardize,
            time_range=time_range,
        )

        if sign_align_ref_conv is not None:
            a0, b0 = xr.align(conv_ts, sign_align_ref_conv, join="inner")
            valid = np.isfinite(a0) & np.isfinite(b0)
            c = xr.corr(a0.where(valid), b0.where(valid), dim=self.cfg.time_name)
            if float(c) < 0.0:
                conv_ts = -conv_ts

        return self._lagcorr_1d(conv_ts, wind_ts, lags)

    # -------- I/O --------
    def save_dataset(self, ds: xr.Dataset, out_nc: Path, *, compute: bool = False) -> None:
        out_nc.parent.mkdir(parents=True, exist_ok=True)
        if compute:
            ds = ds.load()
        ds.to_netcdf(out_nc)

In [17]:
if __name__ == "__main__":
    diag_dir = "/lcrc/group/e3sm/ac.szhang/acme_scratch/e3sm_project/v3_dart_paper"
    out_dir = Path(diag_dir) / "figure_data" / "mjo"
    out_dir.mkdir(parents=True, exist_ok=True)

    exp_list = ["CTRL", "DARTOP", "ERA5OP", "ERA5DL", "ERA5BL", "DARTBL", "DARTBA", "DARTAE"]

    period = "2012-2013"
    start, end = "2012-01-01", "2013-12-31"
    time_slice: Tuple[str, str] = (start, end)

    # Domain / filter settings
    lat_band = (-10.0, 10.0)       # 10S–10N
    lon_sector = (60.0, 150.0)     # 60E–150E (preprocessor normalizes lon to 0..360)
    bandpass_days = (20.0, 100.0)  # 20–100 days

    # -----------------------
    # Reference data (local paths) keyed by MODEL VARIABLE NAME
    # -----------------------
    ref_dict = {
        "FLUT": {
            "name": "NOAA_OLR",
            "path": "/lcrc/group/e3sm/ac.szhang/acme_scratch/e3sm_project/E3SMv3_testings/NOAA-OLR",
            "sub": "post/atm/180x360_aave/ts/daily/1yr",
            "period": period,
            "var": "FLUT",
            "scale": 1.0,
            "sign": +1,
        },
        "U850": {
            "name": "ERA5",
            "path": "/lcrc/group/e3sm/ac.szhang/acme_scratch/e3sm_project/E3SMv3_testings/ERA5",
            "sub": "post/atm/180x360_aave/ts/daily/1yr",
            "period": period,
            "var": "U850",
            "scale": 1.0,
            "sign": +1,
        },
    }

    # -----------------------
    # Model experiment fields (via run_dict)
    # -----------------------
    cfgs = {
        "FLUT": {
            "run_key": "native",
            "sub": "post/atm/180x360_aave/ts/daily/1yr",
            "var": "FLUT",
            "is_cmip6": False,
            "scale": 1.0,
            "sign": +1,
        },
        "U850": {
            "run_key": "native",
            "sub": "post/atm/180x360_aave/ts/daily/1yr",
            "var": "U850",
            "is_cmip6": False,
            "scale": 1.0,
            "sign": +1,
        },
    }

    # -----------------------
    # Instantiate engine
    # -----------------------
    mjo = MJOMetrics(
        cfg=MjoConfig(
            time_name="time",
            lat_name="lat",
            lon_name="lon",
            lat_band=lat_band,
            lon_sector=lon_sector,
            bandpass_days=bandpass_days,
        )
    )

    # -----------------------
    # Loaders
    # -----------------------
    def load_da_model(exp_key: str, var_key: str) -> xr.DataArray:
        vcfg = cfgs[var_key]
        run_key = vcfg["run_key"]

        exp_run = dict(run_dict[exp_key])
        exp_run[run_key] = vcfg["sub"]

        files = find_nc_files(exp_run, run_key, vcfg["var"], is_cmip6=vcfg["is_cmip6"])
        if not files:
            raise FileNotFoundError(f"No files for {exp_key}:{var_key} -> {run_key}:{vcfg['sub']}")

        da = open_var_from_files(files, vcfg["var"])
        da = mjo._safe_time_slice(da, time_slice[0], time_slice[1])
        da = da * vcfg.get("scale", 1.0) * vcfg.get("sign", +1)
        return da

    def load_da_ref(var_key: str) -> xr.DataArray:
        rcfg = ref_dict[var_key]
        pattern = f"{var_key}_*"
        base = Path(rcfg["path"]) / rcfg["sub"]
        files = sorted(glob.glob(str(base / pattern)))
        if not files:
            raise FileNotFoundError(f"No ref files matched: {base / pattern}")

        print(f"\n=== {rcfg['name']} ({rcfg['period']}) ===")
        print(f"{var_key}: nfiles={len(files)} -> {[Path(f).name for f in files]}")

        da = open_var_from_files(files, rcfg["var"])
        da = mjo._safe_time_slice(da, time_slice[0], time_slice[1])
        da = da * rcfg.get("scale", 1.0) * rcfg.get("sign", +1)
        return da

    # -----------------------
    # Load truth once
    # -----------------------
    ref_olr = load_da_ref("FLUT")
    ref_u850 = load_da_ref("U850")

    # -----------------------
    # COUPLING METRICS (restartable, one file per experiment)
    # -----------------------
    print("\n=== MJO Coupling (OLR–U850) ===")

    # build reference convection TS once for sign consistency
    ref_conv_ts = mjo.prep_ts_only(
        ref_olr,
        lon_sector=mjo.cfg.lon_sector,
        bandpass_days=mjo.cfg.bandpass_days,
        standardize=True,
        time_range=time_slice,
    )

    # REF first
    ref_curve = mjo.coupling_from_da(
        ref_olr,
        ref_u850,
        lon_sector=mjo.cfg.lon_sector,
        bandpass_days=mjo.cfg.bandpass_days,
        lags=mjo.cfg.lags,
        standardize=True,
        time_range=time_slice,
        sign_align_ref_conv=None,
    )

    out_nc = out_dir / f"mjo_coupling_OLR_U850_REF_{period}.nc"
    mjo.save_dataset(xr.Dataset({"REF_OLR_ERA5": ref_curve}), out_nc, compute=True)
    print(f"    wrote {out_nc}")

    # Experiments one-by-one (fast restart)
    for exp in exp_list:
        out_nc = out_dir / f"mjo_coupling_OLR_U850_{exp}_{period}.nc"
        if out_nc.exists():
            print(f"  coupling for {exp}: exists -> skip")
            continue

        print(f"  coupling for {exp} ({run_dict[exp]['name']})")

        flut = load_da_model(exp, "FLUT")
        u850 = load_da_model(exp, "U850")

        # Slice before flip test so flip decision matches the coupling window
        flut_s = mjo._safe_time_slice(flut, start, end)
        ref_s = mjo._safe_time_slice(ref_olr, start, end)

        do_flip = should_flip_to_match_ref(mjo, flut_s, ref_s, anomaly="daily_climatology")
        if do_flip:
            flut = -flut

        curve = mjo.coupling_from_da(
            flut,
            u850,
            lon_sector=mjo.cfg.lon_sector,
            bandpass_days=mjo.cfg.bandpass_days,
            lags=mjo.cfg.lags,
            standardize=True,
            time_range=time_slice,
            sign_align_ref_conv=ref_conv_ts,
        )

        mjo.save_dataset(xr.Dataset({exp: curve}), out_nc, compute=True)
        print(f"    wrote {out_nc}")
        


=== NOAA_OLR (2012-2013) ===
FLUT: nfiles=2 -> ['FLUT_201201_201212.nc', 'FLUT_201301_201312.nc']

=== ERA5 (2012-2013) ===
U850: nfiles=2 -> ['U850_201201_201212.nc', 'U850_201301_201312.nc']

=== MJO Coupling (OLR–U850) ===


/lcrc/group/e3sm/ac.szhang/acme_scratch/e3sm_project/v3_dart_paper/mjo_analysis/ExperimentUtil.py:143: FutureWarning: In a future version of xarray the default value for data_vars will change from data_vars='all' to data_vars=None. This is likely to lead to different results when multiple datasets have matching variables with overlapping values. To opt in to new defaults and get rid of these warnings now use `set_options(use_new_combine_kwarg_defaults=True) or set data_vars explicitly.
  ds = xr.open_mfdataset(files, combine="by_coords")


    wrote /lcrc/group/e3sm/ac.szhang/acme_scratch/e3sm_project/v3_dart_paper/figure_data/mjo/mjo_coupling_OLR_U850_REF_2012-2013.nc
  coupling for CTRL (20260114.v3-LR-AMIP-CTRL1.ne30pg2_r05_IcoswISC30E3r5)


/lcrc/group/e3sm/ac.szhang/acme_scratch/e3sm_project/v3_dart_paper/mjo_analysis/ExperimentUtil.py:143: FutureWarning: In a future version of xarray the default value for data_vars will change from data_vars='all' to data_vars=None. This is likely to lead to different results when multiple datasets have matching variables with overlapping values. To opt in to new defaults and get rid of these warnings now use `set_options(use_new_combine_kwarg_defaults=True) or set data_vars explicitly.
  ds = xr.open_mfdataset(files, combine="by_coords")
/lcrc/group/e3sm/ac.szhang/acme_scratch/e3sm_project/v3_dart_paper/mjo_analysis/ExperimentUtil.py:143: FutureWarning: In a future version of xarray the default value for data_vars will change from data_vars='all' to data_vars=None. This is likely to lead to different results when multiple datasets have matching variables with overlapping values. To opt in to new defaults and get rid of these warnings now use `set_options(use_new_combine_kwarg_defaults=

    wrote /lcrc/group/e3sm/ac.szhang/acme_scratch/e3sm_project/v3_dart_paper/figure_data/mjo/mjo_coupling_OLR_U850_CTRL_2012-2013.nc
  coupling for DARTOP (20260114.v3-LR-AMIP-DART1.ne30pg2_r05_IcoswISC30E3r5)


...

    wrote /lcrc/group/e3sm/ac.szhang/acme_scratch/e3sm_project/v3_dart_paper/figure_data/mjo/mjo_coupling_OLR_U850_DARTBL_2012-2013.nc
  coupling for DARTBA (20260114.v3-LR-AMIP-DARTBA.ne30pg2_r05_IcoswISC30E3r5)


/lcrc/group/e3sm/ac.szhang/acme_scratch/e3sm_project/v3_dart_paper/mjo_analysis/ExperimentUtil.py:143: FutureWarning: In a future version of xarray the default value for data_vars will change from data_vars='all' to data_vars=None. This is likely to lead to different results when multiple datasets have matching variables with overlapping values. To opt in to new defaults and get rid of these warnings now use `set_options(use_new_combine_kwarg_defaults=True) or set data_vars explicitly.
  ds = xr.open_mfdataset(files, combine="by_coords")
/lcrc/group/e3sm/ac.szhang/acme_scratch/e3sm_project/v3_dart_paper/mjo_analysis/ExperimentUtil.py:143: FutureWarning: In a future version of xarray the default value for data_vars will change from data_vars='all' to data_vars=None. This is likely to lead to different results when multiple datasets have matching variables with overlapping values. To opt in to new defaults and get rid of these warnings now use `set_options(use_new_combine_kwarg_defaults=

    wrote /lcrc/group/e3sm/ac.szhang/acme_scratch/e3sm_project/v3_dart_paper/figure_data/mjo/mjo_coupling_OLR_U850_DARTBA_2012-2013.nc
  coupling for DARTAE (20260114.v3-LR-AMIP-DARTAE.ne30pg2_r05_IcoswISC30E3r5)


/lcrc/group/e3sm/ac.szhang/acme_scratch/e3sm_project/v3_dart_paper/mjo_analysis/ExperimentUtil.py:143: FutureWarning: In a future version of xarray the default value for data_vars will change from data_vars='all' to data_vars=None. This is likely to lead to different results when multiple datasets have matching variables with overlapping values. To opt in to new defaults and get rid of these warnings now use `set_options(use_new_combine_kwarg_defaults=True) or set data_vars explicitly.
  ds = xr.open_mfdataset(files, combine="by_coords")
/lcrc/group/e3sm/ac.szhang/acme_scratch/e3sm_project/v3_dart_paper/mjo_analysis/ExperimentUtil.py:143: FutureWarning: In a future version of xarray the default value for data_vars will change from data_vars='all' to data_vars=None. This is likely to lead to different results when multiple datasets have matching variables with overlapping values. To opt in to new defaults and get rid of these warnings now use `set_options(use_new_combine_kwarg_defaults=

    wrote /lcrc/group/e3sm/ac.szhang/acme_scratch/e3sm_project/v3_dart_paper/figure_data/mjo/mjo_coupling_OLR_U850_DARTAE_2012-2013.nc
